# Day 2: Managing What the Agent Sees

On Day 1 we built an agent loop. Today we look at **what the agent sees** at each step of that loop.

An LLM only knows what we put in front of it. Everything we send (instructions, past chat, documents, tool results) is called the **context**. The context has a size limit, so we have to choose carefully what goes in.

**By the end of today you will build a pipeline that:**
1. Summarizes old conversation so it doesn't pile up
2. Keeps only the documents that are useful for the question
3. Places the most important instructions at the start and the end
4. Logs how much space each part uses
5. Stays under a fixed size limit, even after 20+ turns

### Before you start

Create a file called `.env` in the same folder as this notebook and add your Azure OpenAI details:

```
AZURE_OPENAI_ENDPOINT=https://<your-resource>.openai.azure.com/
AZURE_OPENAI_API_KEY=<your-key>
AZURE_OPENAI_DEPLOYMENT=<your-chat-deployment-name>
AZURE_OPENAI_API_VERSION=2024-10-21
```

Never share this file or upload it to GitHub, because it contains your secret key.

### Setup
This cell loads your `.env` values and creates the `llm` object we use in the rest of this notebook. Run it first.

In [ ]:
# pip install -U langchain langgraph langchain-openai python-dotenv

import os
from typing import TypedDict
from dotenv import load_dotenv

from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END

# 1. Load the values from the .env file
load_dotenv()

# 2. Make sure nothing important is missing
required = ["AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_KEY", "AZURE_OPENAI_DEPLOYMENT"]
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise ValueError(f"Please add these to your .env file: {missing}")

# 3. Create the LLM
llm = AzureChatOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
    temperature=0,
)

print("Setup done. Using deployment:", os.environ["AZURE_OPENAI_DEPLOYMENT"])

In [ ]:
# Quick check: can we talk to the model?
print(llm.invoke("Say hello to the class in one short sentence.").content)

## 1. Prompt vs. context

A **prompt** is one message you write. The **context** is everything the model sees at once: instructions, earlier chat, documents, and the new question.

An agent rebuilds its context **every turn**. Below, the template has four slots. We fill each slot with fresh values every time we call it.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "{instructions}"),
    ("system", "Earlier conversation:\n{history}"),
    ("system", "Useful documents:\n{docs}"),
    ("human", "{question}"),
])

chain = prompt | llm | StrOutputParser()

answer = chain.invoke({
    "instructions": "You are a travel assistant. Answer in 2 sentences.",
    "history": "User: I'm planning a trip to Goa.\nAssistant: Great! When are you going?",
    "docs": "Goa's monsoon season runs from June to September.",
    "question": "Is July a good time to visit?",
})
print(answer)

Notice that the model answered using the history (Goa) **and** the document (monsoon). Try removing the `docs` value and run it again. The answer gets weaker, because the model is missing the one fact that mattered.

## 2. Counting tokens and setting a budget

Models measure text in **tokens**, not words. A simple rule of thumb is that **1 token is about 4 characters**.

Think of the context like a monthly budget. We decide in advance how much space each part is allowed to use.

In [ ]:
def count_tokens(text):
    """Rough rule: 1 token is about 4 characters."""
    return len(text) // 4

# How much space each part of the context is allowed to use
BUDGET = {"instructions": 60, "summary": 150, "docs": 100, "history": 300}
TOTAL_BUDGET = sum(BUDGET.values())

print("Tokens in a sample sentence:", count_tokens("Is July a good time to visit Goa?"))
print("Total budget:", TOTAL_BUDGET, "tokens")

## 3. Three small helpers

We need three tools to stay inside the budget:

- **`summarize_text`** squeezes old chat into a few lines, using the LLM.
- **`filter_docs`** keeps only documents that share a keyword with the question.
- **`cut_text`** is a safety net. If something is still too long, it gets cut.

In [ ]:
def summarize_text(text):
    """Ask the LLM to compress old conversation into a short summary."""
    return llm.invoke(f"Summarize this conversation in max 5 short lines:\n{text}").content


STOP_WORDS = {"the", "and", "for", "can", "how", "what", "when", "which",
              "why", "you", "tell", "about", "best", "good", "time"}

def filter_docs(question, docs):
    """Keep only documents that share a keyword with the question."""
    words = question.lower().replace("?", "").split()
    keywords = [w for w in words if len(w) >= 3 and w not in STOP_WORDS]
    return [doc for doc in docs if any(k in doc.lower() for k in keywords)]


def cut_text(text, max_tokens):
    """Hard-cut text so it never goes over its budget."""
    return text[: max_tokens * 4]


# Try the filter: only the Goa document should come back
print(filter_docs("When is monsoon in Goa?", [
    "Goa's monsoon season runs from June to September.",
    "Jaipur is called the Pink City.",
]))

## 4. Our knowledge and instructions

This small list stands in for a real document search. `REMINDER` repeats the key rule at the **end** of the prompt.

Why repeat it? Models pay the most attention to the start and the end of a prompt, and less to the middle. This is called **"lost in the middle."**

In [ ]:
KNOWLEDGE = [
    "Goa's monsoon season runs from June to September; beaches are quieter then.",
    "Hyderabad is famous for biryani, the Charminar and Golconda Fort.",
    "Kerala backwater houseboat trips are best between October and March.",
    "Jaipur is called the Pink City and is known for Amber Fort.",
    "Ladakh roads usually open from May to October because of snow.",
    "Indian trains can be booked on IRCTC up to 60 days in advance.",
]

INSTRUCTIONS = "You are a friendly India travel assistant. Answer in under 3 sentences."
REMINDER = "Remember: answer in under 3 sentences."   # repeated at the END of the prompt
KEEP_LAST = 6                                        # recent chat lines kept word-for-word

## 5. The context pipeline as a LangGraph graph

A **LangGraph graph** is a set of steps (called **nodes**) connected by arrows (called **edges**). All nodes share one **state**, a dictionary that each node can read and update.

Our pipeline has three nodes in a straight line:

```
compress_history  ->  build_context  ->  answer_question
```

| Node | What it does |
|---|---|
| `compress_history` | If the chat is long, summarize the old part and keep only the recent lines |
| `build_context` | Pick useful docs, apply the budget, log sizes, and write the final prompt |
| `answer_question` | Send the prompt to the LLM and save the new turn to history |

In [ ]:
class ChatState(TypedDict):
    question: str
    history: list      # recent chat lines, e.g. "User: hi"
    summary: str       # short summary of older chat
    context: str       # final prompt sent to the LLM
    tokens_used: int
    answer: str


# Node 1: summarize old history
def compress_history(state: ChatState):
    history = state["history"]
    if len(history) <= KEEP_LAST:
        return {}                                   # nothing to do yet
    old = history[:-KEEP_LAST]
    recent = history[-KEEP_LAST:]
    new_summary = summarize_text(state["summary"] + "\n" + "\n".join(old))
    print("   (summarized", len(old), "old lines)")
    return {"summary": new_summary, "history": recent}


# Node 2: build the context within budget
def build_context(state: ChatState):
    docs = filter_docs(state["question"], KNOWLEDGE)

    parts = {
        "instructions": INSTRUCTIONS + " " + REMINDER,
        "summary": cut_text(state["summary"], BUDGET["summary"]),
        "docs": cut_text("\n".join(docs), BUDGET["docs"]),
        "history": cut_text("\n".join(state["history"]), BUDGET["history"]),
    }

    # Log how much space each part uses
    sizes = {name: count_tokens(text) for name, text in parts.items()}
    print("   tokens:", sizes, "| total =", sum(sizes.values()))

    # Important instructions at the START and the END
    context = f"""{INSTRUCTIONS}

Summary of earlier chat:
{parts['summary']}

Recent chat:
{parts['history']}

Documents:
{parts['docs']}

Question: {state['question']}

{REMINDER}"""
    return {"context": context, "tokens_used": sum(sizes.values())}


# Node 3: ask the LLM and save the turn
def answer_question(state: ChatState):
    answer = llm.invoke(state["context"]).content
    new_history = state["history"] + [f"User: {state['question']}", f"Assistant: {answer}"]
    return {"answer": answer, "history": new_history}


graph = StateGraph(ChatState)
graph.add_node("compress_history", compress_history)
graph.add_node("build_context", build_context)
graph.add_node("answer_question", answer_question)

graph.add_edge(START, "compress_history")
graph.add_edge("compress_history", "build_context")
graph.add_edge("build_context", "answer_question")
graph.add_edge("answer_question", END)

app = graph.compile()
print("Graph ready: compress_history -> build_context -> answer_question")

## 6. Hands-on lab: a 22-turn conversation

Now we chat for 22 turns and watch the log. Look out for three things:

- The `(summarized ...)` line appears once the chat gets long.
- The token numbers stay steady instead of growing forever.
- Turn 14 asks *"Which city did I ask about first?"* That answer comes from the **summary**, because the original lines are gone.

In [ ]:
questions = [
    "Hi! I'm planning trips around India.", "When is monsoon in Goa?",
    "What is Hyderabad famous for?", "Which fort should I see in Hyderabad?",
    "When should I do a Kerala houseboat trip?", "Why is Jaipur called the Pink City?",
    "What is the main fort in Jaipur?", "When are Ladakh roads open?",
    "How early can I book trains on IRCTC?", "Suggest a 3-day plan for Goa.",
    "What food should I try in Hyderabad?", "Is October good for Kerala?",
    "Can I visit Ladakh in December?", "Which city did I ask about first?",
    "Give me one packing tip for Ladakh.", "Is Goa crowded in July?",
    "What is Amber Fort?", "Recommend a city for history lovers.",
    "Is Hyderabad hot in May?", "What is Charminar?",
    "Summarize all the places we discussed.", "Which trip would you pick for November?",
]

history, summary = [], ""
max_used = 0

for i, q in enumerate(questions, 1):
    print(f"\nTurn {i}: {q}")
    result = app.invoke({"question": q, "history": history, "summary": summary})
    history, summary = result["history"], result["summary"]
    max_used = max(max_used, result["tokens_used"])
    print("   Answer:", result["answer"][:100])

print("\nHighest tokens used in any turn:", max_used, "/", TOTAL_BUDGET)
print("PASS: stayed under budget" if max_used <= TOTAL_BUDGET else "FAIL: over budget")

## Key takeaways

- **Context is rebuilt every turn.** A great prompt still fails if the context around it is messy.
- **Give every part a budget**, and cut or summarize when it gets tight.
- **Old, unused text makes the agent worse** (this is called *context rot*), so remove finished steps.
- **Put key instructions at the start and the end**, not buried in the middle.

**Try this:** Set `KEEP_LAST = 2` and run the lab again. What happens to the summary, and does the agent still answer turn 14 correctly?